# VOC and KITTI YOLOv8n/YOLOv8l benchmarks

Run cells in order. Each engine is evaluated on its full validation split with Ultralytics `val()` using batch 1 and 640-pixel input. Size is the `.engine` file size in decimal MB. Latency measures batch-1 `predict()` on 100 preloaded validation images after 10 warmups. It includes preprocessing, inference and NMS but excludes disk I/O. P95 is the 95th percentile of per-image latency; FPS is `1000 / mean latency in ms`.
ONNX rows use ONNX Runtime CUDAExecutionProvider; TensorRT engine rows use TensorRT. Their latency values depend on different execution backends.


In [1]:
from pathlib import Path
from time import perf_counter
import json

import cv2
import numpy as np
import torch
from ultralytics import YOLO, settings
import ultralytics.data.utils as data_utils

def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / 'original').is_dir() and (folder / 'PTQ').is_dir():
            return folder.resolve()
    raise FileNotFoundError('Cannot locate the Dissertation project root')

PROJECT_ROOT = find_project_root()
ENGINE = PROJECT_ROOT / 'original/yolov8n_voc/voc_8n_best_fp32.engine'
DATASETS_DIR = PROJECT_ROOT / 'PTQ/datasets'
IMAGE_DIR = DATASETS_DIR / 'VOC/images/test2007'
RESULTS_FILE = PROJECT_ROOT / 'PTQ/run_results.json'
DATA = 'VOC.yaml'
DEVICE = 0
IMGSZ = 640
LATENCY_IMAGES = 100
WARMUP_RUNS = 10
CONF = 0.25
IOU = 0.70

if not ENGINE.is_file():
    raise FileNotFoundError(ENGINE)
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(IMAGE_DIR)
if not torch.cuda.is_available():
    raise RuntimeError('TensorRT validation requires CUDA-enabled PyTorch and an NVIDIA GPU')
settings.update({'datasets_dir': str(DATASETS_DIR)})
data_utils.DATASETS_DIR = DATASETS_DIR  # Covers a kernel that imported Ultralytics before this cell.
print('Engine:', ENGINE)
print('Dataset:', IMAGE_DIR)
print('GPU:', torch.cuda.get_device_name(DEVICE))

Engine: C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_voc\voc_8n_best_fp32.engine
Dataset: C:\Users\dxu1\Desktop\Dissertation\PTQ\datasets\VOC\images\test2007
GPU: NVIDIA RTX A2000


In [2]:
model = YOLO(str(ENGINE), task='detect')
metrics = model.val(
    data=DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='voc_8n_best_fp32_engine_val',
    exist_ok=True,
)
ap50 = float(metrics.box.map50)
ap50_95 = float(metrics.box.map)
print(f'AP50: {ap50:.6f}')
print(f'AP50-95: {ap50_95:.6f}')

Ultralytics 8.4.163  Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA RTX A2000, 6138MiB)
Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_voc\voc_8n_best_fp32.engine for TensorRT inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 402.1243.8 MB/s, size: 91.3 KB)
val: Scanning C:\Users\dxu1\Desktop\Dissertation\PTQ\datasets\VOC\labels\test2007.cache... 4952 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4952/4952 1.2Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4952/4952 113.1it/s 43.8s0.1ss
                   all       4952      12032      0.817      0.758      0.825      0.621
Speed: 0.4ms preprocess, 2.8ms inference, 0.0ms loss, 1.2ms postprocess per image
AP50: 0.825078
AP50-95: 0.621306


In [ ]:
image_paths = sorted(IMAGE_DIR.glob('*.jpg'))[:LATENCY_IMAGES]
if not image_paths:
    raise FileNotFoundError(f'No JPEG validation images in {IMAGE_DIR}')
images = [cv2.imread(str(path)) for path in image_paths]  # OpenCV BGR arrays; disk I/O is outside timing.
if any(image is None for image in images):
    raise RuntimeError('At least one validation image could not be read')

for index in range(WARMUP_RUNS):
    model.predict(images[index % len(images)], imgsz=IMGSZ, device=DEVICE,
                  conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

latencies_ms = []
for image in images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                  save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    latencies_ms.append((perf_counter() - start) * 1000.0)

mean_ms = float(np.mean(latencies_ms))
result = {
    'model': str(ENGINE.relative_to(PROJECT_ROOT)),
    'data': DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': ap50,
    'ap50_95': ap50_95,
    'size_mb': ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': mean_ms,
    'latency_p95_ms': float(np.percentile(latencies_ms, 95)),
    'fps': 1000.0 / mean_ms,
    'latency_samples': len(latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(result, indent=2, ensure_ascii=False))


## KITTI YOLOv8n FP32 TensorRT

The available file is `original/yolov8n_kitti/8n_float32_fp32.engine`. The first KITTI validation run downloads the official KITTI dataset if it is not already in `PTQ/datasets`.

In [ ]:
KITTI_ENGINE = PROJECT_ROOT / 'original/yolov8n_kitti/8n_float32_fp32.engine'
KITTI_DATA = 'kitti.yaml'
KITTI_IMAGE_DIR = DATASETS_DIR / 'kitti/images/val'
if not KITTI_ENGINE.is_file():
    raise FileNotFoundError(KITTI_ENGINE)

kitti_model = YOLO(str(KITTI_ENGINE), task='detect')
kitti_metrics = kitti_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8n_float32_fp32_engine_val',
    exist_ok=True,
)
kitti_ap50 = float(kitti_metrics.box.map50)
kitti_ap50_95 = float(kitti_metrics.box.map)
print(f'KITTI AP50: {kitti_ap50:.6f}')
print(f'KITTI AP50-95: {kitti_ap50_95:.6f}')

In [ ]:
kitti_paths = sorted(KITTI_IMAGE_DIR.glob('*.png'))[:LATENCY_IMAGES]
if not kitti_paths:
    kitti_paths = sorted(KITTI_IMAGE_DIR.glob('*.jpg'))[:LATENCY_IMAGES]
if not kitti_paths:
    raise FileNotFoundError(f'No KITTI validation images in {KITTI_IMAGE_DIR}')
kitti_images = [cv2.imread(str(path)) for path in kitti_paths]
if any(image is None for image in kitti_images):
    raise RuntimeError('At least one KITTI image could not be read')

for index in range(WARMUP_RUNS):
    kitti_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                        device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

kitti_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    kitti_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                        save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    kitti_latencies_ms.append((perf_counter() - start) * 1000.0)

kitti_mean_ms = float(np.mean(kitti_latencies_ms))
kitti_result = {
    'model': str(KITTI_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': kitti_ap50,
    'ap50_95': kitti_ap50_95,
    'size_mb': KITTI_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': kitti_mean_ms,
    'latency_p95_ms': float(np.percentile(kitti_latencies_ms, 95)),
    'fps': 1000.0 / kitti_mean_ms,
    'latency_samples': len(kitti_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(kitti_result, indent=2, ensure_ascii=False))

## FP16 KITTI TensorRT

Engine: `original/yolov8n_kitti/8n_float32_fp16.engine`. Use the same validation and latency settings as FP32.


In [ ]:
FP16_ENGINE = PROJECT_ROOT / 'original/yolov8n_kitti/8n_float32_fp16.engine'
if not FP16_ENGINE.is_file():
    raise FileNotFoundError(FP16_ENGINE)

fp16_model = YOLO(str(FP16_ENGINE), task='detect')
fp16_metrics = fp16_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8n_float32_fp16_engine_val',
    exist_ok=True,
)
fp16_ap50 = float(fp16_metrics.box.map50)
fp16_ap50_95 = float(fp16_metrics.box.map)
print(f'KITTI FP16 AP50: {fp16_ap50:.6f}')
print(f'KITTI FP16 AP50-95: {fp16_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the FP32 latency cell.
for index in range(WARMUP_RUNS):
    fp16_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                       device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

fp16_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    fp16_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                       save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    fp16_latencies_ms.append((perf_counter() - start) * 1000.0)

fp16_mean_ms = float(np.mean(fp16_latencies_ms))
fp16_result = {
    'model': str(FP16_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': fp16_ap50,
    'ap50_95': fp16_ap50_95,
    'size_mb': FP16_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': fp16_mean_ms,
    'latency_p95_ms': float(np.percentile(fp16_latencies_ms, 95)),
    'fps': 1000.0 / fp16_mean_ms,
    'latency_samples': len(fp16_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(fp16_result, indent=2, ensure_ascii=False))


## INT8 KITTI TensorRT

Engine: `original/yolov8n_kitti/8n_float32_int8.engine`. Use the same validation and latency settings as FP32 and FP16.


In [ ]:
INT8_ENGINE = PROJECT_ROOT / 'original/yolov8n_kitti/8n_float32_int8.engine'
if not INT8_ENGINE.is_file():
    raise FileNotFoundError(INT8_ENGINE)

int8_model = YOLO(str(INT8_ENGINE), task='detect')
int8_metrics = int8_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8n_float32_int8_engine_val',
    exist_ok=True,
)
int8_ap50 = float(int8_metrics.box.map50)
int8_ap50_95 = float(int8_metrics.box.map)
print(f'KITTI INT8 AP50: {int8_ap50:.6f}')
print(f'KITTI INT8 AP50-95: {int8_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the FP32 latency cell.
for index in range(WARMUP_RUNS):
    int8_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                       device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

int8_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    int8_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                       save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    int8_latencies_ms.append((perf_counter() - start) * 1000.0)

int8_mean_ms = float(np.mean(int8_latencies_ms))
int8_result = {
    'model': str(INT8_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': int8_ap50,
    'ap50_95': int8_ap50_95,
    'size_mb': INT8_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': int8_mean_ms,
    'latency_p95_ms': float(np.percentile(int8_latencies_ms, 95)),
    'fps': 1000.0 / int8_mean_ms,
    'latency_samples': len(int8_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(int8_result, indent=2, ensure_ascii=False))


## YOLOv8l FP32 KITTI TensorRT

Engine: `original/yolov8l_kitti/8l_float32_fp32.engine`. Use the same validation and latency settings as the YOLOv8n KITTI engines.


In [ ]:
KITTI_8L_FP32_ENGINE = PROJECT_ROOT / 'original/yolov8l_kitti/8l_float32_fp32.engine'
if not KITTI_8L_FP32_ENGINE.is_file():
    raise FileNotFoundError(KITTI_8L_FP32_ENGINE)

kitti_8l_fp32_model = YOLO(str(KITTI_8L_FP32_ENGINE), task='detect')
kitti_8l_fp32_metrics = kitti_8l_fp32_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8l_float32_fp32_engine_val',
    exist_ok=True,
)
kitti_8l_fp32_ap50 = float(kitti_8l_fp32_metrics.box.map50)
kitti_8l_fp32_ap50_95 = float(kitti_8l_fp32_metrics.box.map)
print(f'KITTI YOLOv8l FP32 AP50: {kitti_8l_fp32_ap50:.6f}')
print(f'KITTI YOLOv8l FP32 AP50-95: {kitti_8l_fp32_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the YOLOv8n FP32 latency cell.
for index in range(WARMUP_RUNS):
    kitti_8l_fp32_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                                device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

kitti_8l_fp32_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    kitti_8l_fp32_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                                save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    kitti_8l_fp32_latencies_ms.append((perf_counter() - start) * 1000.0)

kitti_8l_fp32_mean_ms = float(np.mean(kitti_8l_fp32_latencies_ms))
kitti_8l_fp32_result = {
    'model': str(KITTI_8L_FP32_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': kitti_8l_fp32_ap50,
    'ap50_95': kitti_8l_fp32_ap50_95,
    'size_mb': KITTI_8L_FP32_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': kitti_8l_fp32_mean_ms,
    'latency_p95_ms': float(np.percentile(kitti_8l_fp32_latencies_ms, 95)),
    'fps': 1000.0 / kitti_8l_fp32_mean_ms,
    'latency_samples': len(kitti_8l_fp32_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(kitti_8l_fp32_result, indent=2, ensure_ascii=False))


## YOLOv8l FP16 KITTI TensorRT

Engine: `original/yolov8l_kitti/8l_float32_fp16.engine`. Use the same validation and latency settings as YOLOv8l FP32.


In [ ]:
KITTI_8L_FP16_ENGINE = PROJECT_ROOT / 'original/yolov8l_kitti/8l_float32_fp16.engine'
if not KITTI_8L_FP16_ENGINE.is_file():
    raise FileNotFoundError(KITTI_8L_FP16_ENGINE)

kitti_8l_fp16_model = YOLO(str(KITTI_8L_FP16_ENGINE), task='detect')
kitti_8l_fp16_metrics = kitti_8l_fp16_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8l_float32_fp16_engine_val',
    exist_ok=True,
)
kitti_8l_fp16_ap50 = float(kitti_8l_fp16_metrics.box.map50)
kitti_8l_fp16_ap50_95 = float(kitti_8l_fp16_metrics.box.map)
print(f'KITTI YOLOv8l FP16 AP50: {kitti_8l_fp16_ap50:.6f}')
print(f'KITTI YOLOv8l FP16 AP50-95: {kitti_8l_fp16_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the YOLOv8n FP32 latency cell.
for index in range(WARMUP_RUNS):
    kitti_8l_fp16_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                                device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

kitti_8l_fp16_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    kitti_8l_fp16_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                                save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    kitti_8l_fp16_latencies_ms.append((perf_counter() - start) * 1000.0)

kitti_8l_fp16_mean_ms = float(np.mean(kitti_8l_fp16_latencies_ms))
kitti_8l_fp16_result = {
    'model': str(KITTI_8L_FP16_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': kitti_8l_fp16_ap50,
    'ap50_95': kitti_8l_fp16_ap50_95,
    'size_mb': KITTI_8L_FP16_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': kitti_8l_fp16_mean_ms,
    'latency_p95_ms': float(np.percentile(kitti_8l_fp16_latencies_ms, 95)),
    'fps': 1000.0 / kitti_8l_fp16_mean_ms,
    'latency_samples': len(kitti_8l_fp16_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(kitti_8l_fp16_result, indent=2, ensure_ascii=False))


## YOLOv8l INT8 KITTI TensorRT

Engine: `original/yolov8l_kitti/8l_float32_int8.engine`. Use the same validation and latency settings as YOLOv8l FP32 and FP16.


In [ ]:
KITTI_8L_INT8_ENGINE = PROJECT_ROOT / 'original/yolov8l_kitti/8l_float32_int8.engine'
if not KITTI_8L_INT8_ENGINE.is_file():
    raise FileNotFoundError(KITTI_8L_INT8_ENGINE)

kitti_8l_int8_model = YOLO(str(KITTI_8L_INT8_ENGINE), task='detect')
kitti_8l_int8_metrics = kitti_8l_int8_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8l_float32_int8_engine_val',
    exist_ok=True,
)
kitti_8l_int8_ap50 = float(kitti_8l_int8_metrics.box.map50)
kitti_8l_int8_ap50_95 = float(kitti_8l_int8_metrics.box.map)
print(f'KITTI YOLOv8l INT8 AP50: {kitti_8l_int8_ap50:.6f}')
print(f'KITTI YOLOv8l INT8 AP50-95: {kitti_8l_int8_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the YOLOv8n FP32 latency cell.
for index in range(WARMUP_RUNS):
    kitti_8l_int8_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                                device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
torch.cuda.synchronize(DEVICE)

kitti_8l_int8_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    kitti_8l_int8_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                                save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    kitti_8l_int8_latencies_ms.append((perf_counter() - start) * 1000.0)

kitti_8l_int8_mean_ms = float(np.mean(kitti_8l_int8_latencies_ms))
kitti_8l_int8_result = {
    'model': str(KITTI_8L_INT8_ENGINE.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': kitti_8l_int8_ap50,
    'ap50_95': kitti_8l_int8_ap50_95,
    'size_mb': KITTI_8L_INT8_ENGINE.stat().st_size / 1_000_000,
    'latency_mean_ms': kitti_8l_int8_mean_ms,
    'latency_p95_ms': float(np.percentile(kitti_8l_int8_latencies_ms, 95)),
    'fps': 1000.0 / kitti_8l_int8_mean_ms,
    'latency_samples': len(kitti_8l_int8_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(kitti_8l_int8_result, indent=2, ensure_ascii=False))


## YOLOv8l INT8 ONNX Runtime KITTI

Model: `original/yolov8l_kitti/8l_float32_int8.int8.onnx`. Validation and latency use the same images and settings as the engines. The runtime is ONNX Runtime CUDAExecutionProvider with CPU fallback, so latency is specific to this backend.


In [ ]:
KITTI_8L_INT8_ONNX = PROJECT_ROOT / 'original/yolov8l_kitti/8l_float32_int8.int8.onnx'
if not KITTI_8L_INT8_ONNX.is_file():
    raise FileNotFoundError(KITTI_8L_INT8_ONNX)

kitti_8l_int8_onnx_model = YOLO(str(KITTI_8L_INT8_ONNX), task='detect')
kitti_8l_int8_onnx_metrics = kitti_8l_int8_onnx_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='8l_float32_int8_onnx_val',
    exist_ok=True,
)
kitti_8l_int8_onnx_ap50 = float(kitti_8l_int8_onnx_metrics.box.map50)
kitti_8l_int8_onnx_ap50_95 = float(kitti_8l_int8_onnx_metrics.box.map)
print(f'KITTI YOLOv8l INT8 ONNX AP50: {kitti_8l_int8_onnx_ap50:.6f}')
print(f'KITTI YOLOv8l INT8 ONNX AP50-95: {kitti_8l_int8_onnx_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the YOLOv8n FP32 latency cell.
for index in range(WARMUP_RUNS):
    kitti_8l_int8_onnx_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                                     device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
onnx_providers = kitti_8l_int8_onnx_model.predictor.model.session.get_providers()
if 'CUDAExecutionProvider' not in onnx_providers:
    raise RuntimeError(f'ONNX Runtime did not use CUDAExecutionProvider: {onnx_providers}')
torch.cuda.synchronize(DEVICE)

kitti_8l_int8_onnx_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    kitti_8l_int8_onnx_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                                     save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    kitti_8l_int8_onnx_latencies_ms.append((perf_counter() - start) * 1000.0)

kitti_8l_int8_onnx_mean_ms = float(np.mean(kitti_8l_int8_onnx_latencies_ms))
kitti_8l_int8_onnx_result = {
    'model': str(KITTI_8L_INT8_ONNX.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'backend': 'ONNX Runtime',
    'providers': onnx_providers,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': kitti_8l_int8_onnx_ap50,
    'ap50_95': kitti_8l_int8_onnx_ap50_95,
    'size_mb': KITTI_8L_INT8_ONNX.stat().st_size / 1_000_000,
    'latency_mean_ms': kitti_8l_int8_onnx_mean_ms,
    'latency_p95_ms': float(np.percentile(kitti_8l_int8_onnx_latencies_ms, 95)),
    'fps': 1000.0 / kitti_8l_int8_onnx_mean_ms,
    'latency_samples': len(kitti_8l_int8_onnx_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(kitti_8l_int8_onnx_result, indent=2, ensure_ascii=False))


## Old YOLOv8n INT8 ONNX Runtime KITTI

Model: `original/yolov8n_kitti/old_8n_float32_int8.onnx`. Validation and latency use the same KITTI images and settings as the other models. The runtime is ONNX Runtime CUDAExecutionProvider with CPU fallback.


In [ ]:
OLD_8N_INT8_ONNX = PROJECT_ROOT / 'original/yolov8n_kitti/old_8n_float32_int8.onnx'
if not OLD_8N_INT8_ONNX.is_file():
    raise FileNotFoundError(OLD_8N_INT8_ONNX)

old_8n_int8_onnx_model = YOLO(str(OLD_8N_INT8_ONNX), task='detect')
old_8n_int8_onnx_metrics = old_8n_int8_onnx_model.val(
    data=KITTI_DATA, split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
    workers=0, plots=False, save_json=False, verbose=False,
    project=str(PROJECT_ROOT / 'PTQ/runs'), name='old_8n_float32_int8_onnx_val',
    exist_ok=True,
)
old_8n_int8_onnx_ap50 = float(old_8n_int8_onnx_metrics.box.map50)
old_8n_int8_onnx_ap50_95 = float(old_8n_int8_onnx_metrics.box.map)
print(f'KITTI Old YOLOv8n INT8 ONNX AP50: {old_8n_int8_onnx_ap50:.6f}')
print(f'KITTI Old YOLOv8n INT8 ONNX AP50-95: {old_8n_int8_onnx_ap50_95:.6f}')


In [ ]:
# Reuse the preloaded KITTI validation images from the YOLOv8n FP32 latency cell.
for index in range(WARMUP_RUNS):
    old_8n_int8_onnx_model.predict(kitti_images[index % len(kitti_images)], imgsz=IMGSZ,
                                     device=DEVICE, conf=CONF, iou=IOU, save=False, verbose=False)
onnx_providers = old_8n_int8_onnx_model.predictor.model.session.get_providers()
if 'CUDAExecutionProvider' not in onnx_providers:
    raise RuntimeError(f'ONNX Runtime did not use CUDAExecutionProvider: {onnx_providers}')
torch.cuda.synchronize(DEVICE)

old_8n_int8_onnx_latencies_ms = []
for image in kitti_images:
    torch.cuda.synchronize(DEVICE)
    start = perf_counter()
    old_8n_int8_onnx_model.predict(image, imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU,
                                     save=False, verbose=False)
    torch.cuda.synchronize(DEVICE)
    old_8n_int8_onnx_latencies_ms.append((perf_counter() - start) * 1000.0)

old_8n_int8_onnx_mean_ms = float(np.mean(old_8n_int8_onnx_latencies_ms))
old_8n_int8_onnx_result = {
    'model': str(OLD_8N_INT8_ONNX.relative_to(PROJECT_ROOT)),
    'data': KITTI_DATA,
    'backend': 'ONNX Runtime',
    'providers': onnx_providers,
    'device': torch.cuda.get_device_name(DEVICE),
    'ap50': old_8n_int8_onnx_ap50,
    'ap50_95': old_8n_int8_onnx_ap50_95,
    'size_mb': OLD_8N_INT8_ONNX.stat().st_size / 1_000_000,
    'latency_mean_ms': old_8n_int8_onnx_mean_ms,
    'latency_p95_ms': float(np.percentile(old_8n_int8_onnx_latencies_ms, 95)),
    'fps': 1000.0 / old_8n_int8_onnx_mean_ms,
    'latency_samples': len(old_8n_int8_onnx_latencies_ms),
    'warmup_runs': WARMUP_RUNS,
    'imgsz': IMGSZ,
    'latency_definition': 'batch-1 predict: preprocess + inference + NMS; disk I/O excluded',
}
print(json.dumps(old_8n_int8_onnx_result, indent=2, ensure_ascii=False))


In [ ]:
results = [result, kitti_result, fp16_result, int8_result, kitti_8l_fp32_result, kitti_8l_fp16_result, kitti_8l_int8_result, kitti_8l_int8_onnx_result, old_8n_int8_onnx_result]
RESULTS_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(f"{'Model':48} {'Backend':16} {'AP50':>8} {'AP50-95':>9} {'Size MB':>9} {'Mean ms':>9} {'P95 ms':>9} {'FPS':>9}")
for row in results:
    print(f"{Path(row['model']).name:48} {row.get('backend', 'TensorRT'):16} {row['ap50']:8.4f} {row['ap50_95']:9.4f} "
          f"{row['size_mb']:9.2f} {row['latency_mean_ms']:9.2f} "
          f"{row['latency_p95_ms']:9.2f} {row['fps']:9.1f}")
print('Saved:', RESULTS_FILE)

## Measured results (NVIDIA RTX A2000)

| Model | Dataset | Backend | AP50 | AP50-95 | Size (MB) | Mean latency (ms) | P95 (ms) | FPS |
|---|---|---|---:|---:|---:|---:|---:|---:|
| `voc_8n_best_fp32.engine` | VOC | TensorRT | 0.8251 | 0.6213 | 93.51 | 4.73 | 5.83 | 211.2 |
| `8n_float32_fp32.engine` | KITTI | TensorRT | 0.9049 | 0.6755 | 79.82 | 5.04 | 6.39 | 198.6 |
| `8n_float32_fp16.engine` | KITTI | TensorRT | 0.9074 | 0.6763 | 53.29 | 4.18 | 5.63 | 239.5 |
| `8n_float32_int8.engine` | KITTI | TensorRT | 0.9008 | 0.6657 | 40.82 | 3.49 | 4.36 | 286.6 |
| `8l_float32_fp32.engine` | KITTI | TensorRT | 0.9466 | 0.7756 | 372.96 | 21.00 | 22.52 | 47.6 |
| `8l_float32_fp16.engine` | KITTI | TensorRT | 0.9468 | 0.7745 | 225.89 | 11.12 | 11.93 | 90.0 |
| `8l_float32_int8.engine` | KITTI | TensorRT | 0.9345 | 0.7527 | 76.62 | 6.40 | 7.46 | 156.2 |
| `8l_float32_int8.int8.onnx` | KITTI | ONNX Runtime | 0.9357 | 0.7507 | 87.83 | 25.34 | 26.84 | 39.5 |
| `old_8n_float32_int8.onnx` | KITTI | ONNX Runtime | 0.8854 | 0.6163 | 3.39 | 10.91 | 12.32 | 91.7 |

AP values use the full validation split. Latency uses 100 preloaded images, 10 warmups and batch 1; it includes preprocessing and NMS but excludes disk I/O. ONNX Runtime uses CUDAExecutionProvider with CPUExecutionProvider fallback; engine rows use TensorRT. Reruns may produce different latency values.
